In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), '..', '00-setup'))
from spark_session import get_spark, output_path, stop_and_wait
from seed_data import load_all, register_views
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = get_spark("29-data-cleaning-workflow")
dfs   = register_views(spark)
emp   = dfs["employees"]
dept  = dfs["departments"]
sal   = dfs["salary_history"]
proj  = dfs["projects"]
pa    = dfs["project_assignments"]
pr    = dfs["performance_reviews"]
ee    = dfs["emp_events"]
po    = dfs["purchase_orders"]
lr    = dfs["leave_requests"]


# ── Section 1 ── Load and profile raw data ────────────────────────────────────
# describe() gives count/mean/stddev/min/max for numeric cols
# Flaws surfaced: salary mean includes 0.0 (emp 19), count < 41 for salary (NULLs excluded)
print("\n── Section 1: Raw profile ──")
emp.describe("emp_id", "salary").show()

# Per-column NULL counts — use reduce-style aggregation across all columns
null_counts = emp.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in emp.columns
])
print("NULL counts per column:")
null_counts.show()
# Expected: salary=2 (emp 10,15), email=1 (emp 22)

print(f"Raw row count: {emp.count()}")     # 41


# ── Section 2 ── Schema validation ────────────────────────────────────────────
# Verify expected dtypes; flag mismatches vs canonical definition
print("\n── Section 2: Schema validation ──")
expected_dtypes = {
    "emp_id":           "int",
    "first_name":       "string",
    "last_name":        "string",
    "email":            "string",
    "phone":            "string",
    "dept_id":          "int",
    "manager_id":       "int",
    "job_title":        "string",
    "salary":           "double",
    "hire_date":        "date",
    "status":           "string",
    "termination_date": "date",
}
actual_dtypes = dict(emp.dtypes)
print(f"{'Column':<20} {'Expected':<12} {'Actual':<12} {'Match'}")
print("-" * 58)
for col, expected in expected_dtypes.items():
    actual  = actual_dtypes.get(col, "MISSING")
    match   = "OK" if actual == expected else "MISMATCH"
    print(f"{col:<20} {expected:<12} {actual:<12} {match}")


# ── Section 3 ── NULL handling ─────────────────────────────────────────────────
# Strategy A: fill NULL salary with department average (window-based imputation)
# Strategy B: fill NULL email with placeholder
# Strategy C: drop rows where BOTH salary AND email are NULL (no usable data)
print("\n── Section 3: NULL handling ──")

w_dept  = Window.partitionBy("dept_id")
emp_w   = emp.withColumn(
    "dept_avg_salary",
    F.avg("salary").over(w_dept)           # NULL rows excluded from avg by Spark
)
emp_imp = emp_w.withColumn(
    "salary",
    F.coalesce(F.col("salary"), F.col("dept_avg_salary"))
).drop("dept_avg_salary")

# Fill NULL email
emp_imp = emp_imp.withColumn(
    "email",
    F.coalesce(F.col("email"), F.lit("unknown@company.com"))
)

# Drop rows where BOTH salary and email were NULL before imputation
# (check on original emp, not imp)
both_null = emp.filter(F.col("salary").isNull() & F.col("email").isNull())
print(f"Rows with both salary AND email NULL (to drop): {both_null.count()}")
# For this dataset = 0; no row has both null — but pattern is still applied defensively
emp_clean = emp_imp.join(both_null.select("emp_id"), on="emp_id", how="left_anti")

# Verify imputation
print("Salary NULL after imputation:")
emp_clean.filter(F.col("salary").isNull()).select("emp_id", "salary").show()
print("email NULL after imputation:")
emp_clean.filter(F.col("email") == "unknown@company.com").select("emp_id", "email").show()


# ── Section 4 ── Duplicate detection ──────────────────────────────────────────
# Detect exact duplicates in salary_history (hist_id differs, biz-key same)
# Detect soft-dups in employees (same dept/hire_date/title → likely same person)
print("\n── Section 4: Duplicate detection ──")

sal_biz = ["emp_id", "salary_before", "salary_after", "effective_date", "change_reason", "changed_by"]
sal_dups = sal.groupBy(sal_biz) \
              .agg(F.count("*").alias("cnt"), F.collect_list("hist_id").alias("hist_ids")) \
              .filter(F.col("cnt") > 1)
print("Exact duplicates in salary_history (biz-key level):")
sal_dups.show(truncate=False)
# Expected: 1 group — emp 5, 2022-04-01

soft_keys = ["last_name", "dept_id", "hire_date", "job_title"]
emp_soft  = emp.groupBy(soft_keys) \
               .agg(
                   F.count("*").alias("cnt"),
                   F.collect_list("emp_id").alias("emp_ids"),
                   F.collect_list("salary").alias("salaries"),
               ).filter(F.col("cnt") > 1)
print("Soft-duplicates in employees (same dept/hire_date/title):")
emp_soft.show(truncate=False)
# Expected: emp 18 and emp 19 (emp 19 has salary=0.0)


# ── Section 5 ── Deduplication ────────────────────────────────────────────────
# salary_history exact-dup: keep row with lowest hist_id (ROW_NUMBER over biz-key)
# employees soft-dup: keep emp with non-zero salary (emp 18 wins over emp 19)
print("\n── Section 5: Deduplication ──")

w_sal_dedup = Window.partitionBy(*sal_biz).orderBy("hist_id")
sal_clean = sal.withColumn("rn", F.row_number().over(w_sal_dedup)) \
               .filter(F.col("rn") == 1).drop("rn")
print(f"salary_history: {sal.count()} rows → {sal_clean.count()} after dedup")
# Expected: 33 → 32

# For soft-dup employees: rank by salary desc (non-zero salary ranks first)
w_soft = Window.partitionBy(*soft_keys).orderBy(F.col("salary").desc_nulls_last())
emp_clean_dedup = emp_clean \
    .withColumn("soft_rn", F.row_number().over(w_soft)) \
    .filter(F.col("soft_rn") == 1).drop("soft_rn")
print(f"employees: {emp_clean.count()} rows → {emp_clean_dedup.count()} after soft-dedup")
# Expected: 41 → 40 (emp 19 removed; emp 18 kept)


# ── Section 6 ── Outlier detection ────────────────────────────────────────────
# Flag salary=0.0 (emp 19 before dedup, data-entry error)
# Flag future hire_date (emp 41 — hire_date=2025-08-01, beyond any reasonable load date)
print("\n── Section 6: Outlier detection ──")

today = F.current_date()
outliers = emp.withColumn(
    "outlier_flag",
    F.when(F.col("salary") == 0.0,         F.lit("salary_zero"))
     .when(F.col("salary").isNull(),        F.lit("salary_null"))
     .when(F.col("hire_date") > today,      F.lit("future_hire_date"))
     .otherwise(F.lit(None).cast("string"))
).filter(F.col("outlier_flag").isNotNull())

print("Outlier employees:")
outliers.select("emp_id", "first_name", "salary", "hire_date", "outlier_flag").show()
# Expected: emp 10 (salary_null), emp 15 (salary_null), emp 19 (salary_zero), emp 41 (future_hire_date)


# ── Section 7 ── Referential integrity ────────────────────────────────────────
# Employees with no matching dept_id in departments table
# purchase_orders with NULL dept_id (row 25 — orphaned)
print("\n── Section 7: Referential integrity ──")

emp_orphan = emp.join(dept, on="dept_id", how="left_anti")
print("Employees with no matching dept_id:")
emp_orphan.select("emp_id", "dept_id").show()
# Expected: 0 (all emp dept_ids exist in departments for this dataset)

po_null_dept = dfs["purchase_orders"].filter(F.col("dept_id").isNull())
print("Purchase orders with NULL dept_id (orphaned):")
po_null_dept.select("order_id", "dept_id", "vendor", "amount").show()
# Expected: 1 row — order_id=25


# ── Section 8 ── Date consistency ─────────────────────────────────────────────
# leave_requests where end_date < start_date
# projects where end_date < start_date (proj 11)
print("\n── Section 8: Date consistency ──")

lr_bad_dates = lr.filter(F.col("end_date") < F.col("start_date"))
print("Leave requests with end_date < start_date:")
lr_bad_dates.select("request_id", "emp_id", "start_date", "end_date").show()
# Expected: 0 for this dataset (leave flaws are overlaps, not end < start)

proj_bad = proj.filter(
    F.col("end_date").isNotNull() & (F.col("end_date") < F.col("start_date"))
)
print("Projects with end_date < start_date:")
proj_bad.select("project_id", "project_name", "start_date", "end_date").show()
# Expected: proj 11


# ── Section 9 ── Overlap detection ────────────────────────────────────────────
# Leave requests for same employee with overlapping date ranges (emp 12 req 1 and 2)
# Self-join: start_a <= end_b AND end_a >= start_b (and not same request)
print("\n── Section 9: Overlap detection ──")

lr_overlap = lr.alias("a").join(
    lr.alias("b"),
    (F.col("a.emp_id")      == F.col("b.emp_id")) &
    (F.col("a.request_id")   < F.col("b.request_id")) &
    (F.col("a.start_date")  <= F.col("b.end_date")) &
    (F.col("a.end_date")    >= F.col("b.start_date"))
).select(
    F.col("a.emp_id").alias("emp_id"),
    F.col("a.request_id").alias("req_1"),
    F.col("a.start_date").alias("start_1"),
    F.col("a.end_date").alias("end_1"),
    F.col("b.request_id").alias("req_2"),
    F.col("b.start_date").alias("start_2"),
    F.col("b.end_date").alias("end_2"),
)
print("Overlapping leave requests:")
lr_overlap.show(truncate=False)
# Expected: emp 12 requests 1 and 2


# ── Section 10 ── Standardize strings ─────────────────────────────────────────
# Trim + lower-case email addresses; trim job_title of leading/trailing whitespace
print("\n── Section 10: Standardize strings ──")

emp_std = emp_clean_dedup \
    .withColumn("email",     F.lower(F.trim(F.col("email")))) \
    .withColumn("job_title", F.trim(F.col("job_title")))

# Spot-check: show a sample of emails and titles
emp_std.select("emp_id", "email", "job_title").limit(8).show(truncate=False)


# ── Section 11 ── Audit columns ───────────────────────────────────────────────
# cleaned_at: timestamp of when the cleaning ran
# cleaning_flags: comma-joined string of issues found per employee row
# Flags: salary_imputed, email_placeholder, salary_zero, future_hire_date
print("\n── Section 11: Audit columns ──")

today_val = F.current_date()

emp_audited = emp_std \
    .withColumn("was_salary_null",   emp.select("emp_id", F.col("salary").isNull().alias("f"))
                                       .join(emp_std.select("emp_id"), "emp_id").select("f") if False
                                       else F.lit(False)) \
    .drop("was_salary_null")

# Rejoin original emp to compare what was imputed
orig_nulls = emp.select(
    "emp_id",
    F.col("salary").isNull().cast("int").alias("orig_sal_null"),
    F.col("email").isNull().cast("int").alias("orig_email_null"),
)
emp_audited = emp_std.join(orig_nulls, on="emp_id", how="left")

emp_audited = emp_audited \
    .withColumn("flag_salary_imputed",   F.when(F.col("orig_sal_null")   == 1, F.lit("salary_imputed")).otherwise(F.lit(None).cast("string"))) \
    .withColumn("flag_email_placeholder",F.when(F.col("orig_email_null") == 1, F.lit("email_placeholder")).otherwise(F.lit(None).cast("string"))) \
    .withColumn("flag_salary_zero",      F.when(F.col("salary") == 0.0,        F.lit("salary_zero")).otherwise(F.lit(None).cast("string"))) \
    .withColumn("flag_future_hire",      F.when(F.col("hire_date") > today_val,F.lit("future_hire_date")).otherwise(F.lit(None).cast("string")))

# Combine all flags into a single comma-joined string; filter out nulls
flag_cols = ["flag_salary_imputed", "flag_email_placeholder", "flag_salary_zero", "flag_future_hire"]

# Build array, filter nulls, join to string
emp_audited = emp_audited \
    .withColumn(
        "cleaning_flags",
        F.concat_ws(
            ",",
            *[F.coalesce(F.col(c), F.lit("")) for c in flag_cols]
        )
    ) \
    .withColumn("cleaning_flags",
        F.when(F.col("cleaning_flags") == "", F.lit(None).cast("string"))
         .otherwise(F.regexp_replace(F.col("cleaning_flags"), "(^,+|,+$|,{2,})", ","))) \
    .withColumn("cleaned_at", F.current_timestamp()) \
    .drop(*flag_cols, "orig_sal_null", "orig_email_null")

print("Employees with cleaning flags:")
emp_audited.filter(F.col("cleaning_flags").isNotNull()) \
           .select("emp_id", "salary", "email", "hire_date", "cleaning_flags", "cleaned_at") \
           .show(truncate=False)
# Expected flags: emp 10,15 → salary_imputed; emp 22 → email_placeholder; emp 41 → future_hire_date


# ── Section 12 ── Write cleaned employees to Parquet ─────────────────────────
# Spark UI → Storage tab: no cached RDD/DF here; check SQL tab for the write plan
# SQL tab → see the Exchange nodes from the window functions in Sections 3 and 11
print("\n── Section 12: Write to Parquet ──")

out = output_path("parquet/employees_cleaned")
emp_audited.write.mode("overwrite").parquet(out)
print(f"Cleaned employees written to: {out}")
print(f"Row count written: {emp_audited.count()}")
# Spark UI → Jobs tab: this .write triggers a new job; click it to see the Stage DAG
# Stages will show: scan → project/filter → exchange (shuffle for window) → write


stop_and_wait(spark)